# DSA 495 - Module 06
## Text Clustering and Topic Modeling with BERTopic

**Student notebook**

This activity discovers themes in **1,398 HuffPost articles from 2022**. Its centerpiece follows the *BERTopic Best Practices* “Visualize Topics” section closely: we will use topic-word bars, a document map, an intertopic distance map, and a topic hierarchy, then test what those pictures suggest by returning to the articles.

The activity uses only the article text and descriptive metadata. No publisher-assigned category column is needed.

## Learning objectives

By the end of class, students should be able to:

1. explain the practical BERTopic pipeline from document embeddings to topic representations;
2. interpret topics using both c-TF-IDF words and representative articles;
3. interpret document groups in a two-dimensional projection;
4. read an intertopic distance map and a topic hierarchy without treating either as ground truth; and
5. explain why HDBSCAN marks some documents as outliers.

## 0. Setup

Run the install cell once in Colab. If Colab asks to restart the runtime, restart it and continue with the import cell.

In [1]:
%pip install -q "bertopic==0.17.4" "sentence-transformers>=3,<7" "pandas>=2.2,<3" "plotly>=6,<8"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import random

import numpy as np
import pandas as pd
import torch
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

pd.set_option("display.max_colwidth", 120)

/opt/hostedtoolcache/Python/3.11.16/x64/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Document collection

The source file is the [News Category Dataset](https://www.kaggle.com/datasets/rmisra/news-category-dataset), a collection of HuffPost articles. The prepared CSV keeps the most recent year available in the local source: **2022**.

Each modeling document combines the headline and short description. The CSV also preserves the date, author, and link so we can return from a cluster to the source context.

In [3]:
# Load the prepared Module 06 dataset directly from this GitHub repo
DATA_URL = "https://raw.githubusercontent.com/StrokeOfLuck/-DSA495-TextAnalysis-06/main/data/news_category_2022.csv"
news = pd.read_csv(DATA_URL, parse_dates=["date"])
print(f"Loaded {len(news):,} rows from GitHub")
news.head()


Loaded 1,398 rows from GitHub


,document_id,date,year,headline,short_description,authors,link,text
0,news-2022-0001,2022-09-23,2022,23 Of The Funniest Tweets About Cats And Dogs This Week (Sept. 17-23),"""Until you have a dog you don't understand what could be eaten.""",Elyse Wanshel,https://www.huffpost.com/entry/funniest-tweets-cats-dogs-september-17-23_n_632de332e4b0695c1d81dc02,"23 Of The Funniest Tweets About Cats And Dogs This Week (Sept. 17-23). ""Until you have a dog you don't understand wh..."
1,news-2022-0002,2022-09-23,2022,"American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video","He was subdued by passengers and crew when he fled to the back of the aircraft after the confrontation, according to...",Mary Papenfuss,https://www.huffpost.com/entry/american-airlines-passenger-banned-flight-attendant-punch-justice-department_n_632e25...,"American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video. He was subdued by passeng..."
2,news-2022-0003,2022-09-23,2022,Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters,Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new b...,"Carla K. Johnson, AP",https://www.huffpost.com/entry/covid-boosters-uptake-us_n_632d719ee4b087fae6feaac9,Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters. Health experts said it is too early to...
3,news-2022-0004,2022-09-23,2022,The Funniest Tweets From Parents This Week (Sept. 17-23),"""Accidentally put grown-up toothpaste on my toddler’s toothbrush and he screamed like I was cleaning his teeth with ...",Caroline Bologna,https://www.huffpost.com/entry/funniest-parenting-tweets_l_632d7d15e4b0d12b5403e479,"The Funniest Tweets From Parents This Week (Sept. 17-23). ""Accidentally put grown-up toothpaste on my toddler’s toot..."
4,news-2022-0005,2022-09-22,2022,Cleaner Was Dead In Belk Bathroom For 4 Days Before Body Found: Police,The 63-year-old woman was seen working at the South Carolina store on Thursday. She was found dead Monday after her ...,NaN,https://www.huffpost.com/entry/belk-worker-found-dead-columbiana-centre-bathroom_n_632c5f8ce4b0572027b0251d,Cleaner Was Dead In Belk Bathroom For 4 Days Before Body Found: Police. The 63-year-old woman was seen working at th...


In [4]:
news[["document_id", "date", "headline"]].head()

,document_id,date,headline
0,news-2022-0001,2022-09-23,23 Of The Funniest Tweets About Cats And Dogs This Week (Sept. 17-23)
1,news-2022-0002,2022-09-23,"American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video"
2,news-2022-0003,2022-09-23,Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters
3,news-2022-0004,2022-09-23,The Funniest Tweets From Parents This Week (Sept. 17-23)
4,news-2022-0005,2022-09-22,Cleaner Was Dead In Belk Bathroom For 4 Days Before Body Found: Police


In [5]:
df_test = news.sample(6, random_state=42)[
    ["headline", "short_description"]
]

In [6]:
df_test

,headline,short_description
78,'Our Hearts Are Broken': Historic Front Pages Mark The Queen's Death,Both British and international newspapers honor the passing of the U.K'.s longest-reigning monarch.
478,Herschel Walker Says There Are 52 States,There are 50.
354,Twitter Lawyer Calls Elon Musk 'Committed Enemy' As Judge Sets October Trial,Delaware Chancery Judge Kathaleen McCormick dealt the world's richest person a setback in ordering a speedy trial on...
594,"Angel Statue Beheaded At Church, $2 Million Relic Stolen",The New York City church says its stolen 18-carat gold relic was guarded by its own security system and is irreplace...
1255,"Howard Hesseman, Star Of ‘WKRP In Cincinnati,’ Dies At 81",The two-time Emmy nominee played the radio disc jockey Johnny Fever on the sitcom “WKRP in Cincinnati.”
993,Top Putin Flack Sergey Lavrov Praises Fox News,"Russia's top diplomat had some warm words for the conservative network, whose prime-time personality Tucker Carlson ..."


### Your turn: predict before modeling

Read the six sampled articles.

- What two or three themes might BERTopic discover?
- Which words in the text support each prediction?
- Which article might be difficult to place in a dense group, and why?

## 2. Build the BERTopic pipeline

![BERTopic pipeline from document embeddings to topic representations](https://maartengr.github.io/BERTopic/algorithm/default.svg)

| Step | Role in this activity |
|---|---|
| Sentence Transformer | Turns each article into a semantic embedding. |
| UMAP | Compresses embeddings so density-based clustering is workable. |
| HDBSCAN | Finds dense groups and may leave some documents as outliers. |
| c-TF-IDF | Finds words that distinguish each cluster from the other clusters. |
| Human interpretation | Checks documents and proposes a defensible topic name. |

BERTopic is modular: we explicitly supply the embedding, UMAP, HDBSCAN, and vectorizer models. BERTopic then calculates the c-TF-IDF topic representation.

In [7]:
documents = news["text"].tolist()

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(1)

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
EMBEDDING_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    revision=EMBEDDING_REVISION,
    device="cpu",
)
embeddings = embedding_model.encode(documents, show_progress_bar=True)

embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3945.62it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Batches:   2%|▏         | 1/44 [00:00<00:25,  1.70it/s]

Batches:   5%|▍         | 2/44 [00:01<00:23,  1.75it/s]

...

Batches:  95%|█████████▌| 42/44 [00:19<00:00,  2.67it/s]

Batches:  98%|█████████▊| 43/44 [00:19<00:00,  2.78it/s]

Batches: 100%|██████████| 44/44 [00:19<00:00,  3.26it/s]

Batches: 100%|██████████| 44/44 [00:19<00:00,  2.24it/s]

(1398, 384)

> **Discussion pause:** There are 1,398 rows in the embedding matrix, one per article. The columns are numerical features learned by the language model; they are not topic labels.

In [8]:
umap_model = UMAP(          # Creates a model for reducing embedding dimensions.
    n_neighbors=12,         # Considers 12 nearby documents when preserving structure.
    n_components=5,         # Reduces each embedding to 5 dimensions.
    min_dist=0.0,           # Allows similar documents to be packed tightly together.
    metric="cosine",        # Measures similarity using vector direction.
    random_state=42,        # Makes results reproducible.
)

vectorizer_model = CountVectorizer(  # Converts topic documents into word-count features.
    stop_words="english",            # Removes common English words.
    ngram_range=(1, 2),              # Uses single words and two-word phrases.
    min_df=3,                        # Keeps terms appearing in at least 3 documents.
)

hdbscan_model = HDBSCAN(             # Groups similar documents into clusters.
    min_cluster_size=25,             # Requires at least 25 documents per cluster.
    min_samples=5,                   # Controls how conservative the clustering is.
    metric="euclidean",              # Uses straight-line distance after UMAP.
    cluster_selection_method="eom",  # Selects stable, persistent clusters.
    prediction_data=True,            # Saves information needed to classify new documents.
    core_dist_n_jobs=1,              # Uses one CPU job for core-distance calculations.
)

topic_model = BERTopic(                       # Creates the complete topic-modeling pipeline.
    embedding_model=embedding_model,          # Converts text into semantic embeddings.
    umap_model=umap_model,                    # Uses the configured dimension-reduction model.
    hdbscan_model=hdbscan_model,              # Uses the configured clustering model.
    vectorizer_model=vectorizer_model,        # Uses the configured word-count model.
    top_n_words=8,                            # Shows the top 8 words for each topic.
    verbose=True,                             # Displays progress while the model runs.
)

In [9]:
topics, probabilities = topic_model.fit_transform(
    documents,   # First argument: the original text documents
    embeddings   # Second argument: the document embedding vectors
)  # Fits BERTopic and assigns a topic and probability to each document.

topic_info = topic_model.get_topic_info() #topic_model is a BERTtopic object
# Creates a summary table containing information about the discovered topics.

topic_info[["Topic", "Count", "Name", "Representation"]]
# Displays selected columns: topic ID, document count, name, and representative words.

2026-09-28 19:57:16,608 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


2026-09-28 19:57:30,087 - BERTopic - Dimensionality - Completed ✓


2026-09-28 19:57:30,088 - BERTopic - Cluster - Start clustering the reduced embeddings


2026-09-28 19:57:30,116 - BERTopic - Cluster - Completed ✓


2026-09-28 19:57:30,118 - BERTopic - Representation - Fine-tuning topics using representation models.


2026-09-28 19:57:30,178 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation
0,-1,238,-1_police_twitter_man_said,"[police, twitter, man, said, new, water, people, woman]"
1,0,252,0_ukraine_russia_russian_biden,"[ukraine, russia, russian, biden, war, president, said, day]"
2,1,226,1_star_actor_new_rock,"[star, actor, new, rock, queen, chris, series, film]"
3,2,138,2_trump_gop_senate_election,"[trump, gop, senate, election, fox, news, fox news, host]"
4,3,112,3_trump_jan_ex_capitol,"[trump, jan, ex, capitol, probe, committee, house, trial]"
5,4,87,4_shooting_school_police_gun,"[shooting, school, police, gun, killed, man, killing, year old]"
6,5,64,5_abortion_court_supreme court_supreme,"[abortion, court, supreme court, supreme, rights, abortion rights, ban, law]"
7,6,64,6_covid_covid 19_19_cases,"[covid, covid 19, 19, cases, health, mandate, new, city]"
8,7,46,7_beijing_winter_team_gold,"[beijing, winter, team, gold, open, title, win, 2022]"
9,8,41,8_dead_missing_crash_killed,"[dead, missing, crash, killed, injured, people, dozens, days]"


In [10]:
len(topic_info)

14

In [11]:
model_summary = pd.Series(
    {
        "non-outlier topics": (topic_info["Topic"] != -1).sum(),
        "documents assigned to topics": (pd.Series(topics) != -1).sum(),
        "outlier documents": (pd.Series(topics) == -1).sum(),
        "outlier share": (pd.Series(topics) == -1).mean(),
    }
)
model_summary

non-outlier topics                13.000000
documents assigned to topics    1160.000000
outlier documents                238.000000
outlier share                      0.170243
dtype: float64

> **Key point:** Topic numbers are identifiers, not rankings. Topic `-1` is HDBSCAN's outlier group: articles that do not sit in a sufficiently dense cluster under these settings. Keep that uncertainty visible.

## 3. Give the plots readable labels

The model's top words provide compact display labels. These labels improve the figures but do not replace interpretation or validation.

In [12]:
readable_topic_labels = topic_model.generate_topic_labels(
    nr_words=3,
    topic_prefix=True,
    separator=" · ",
)

readable_topic_labels[0] = "Outliers"
topic_model.set_topic_labels(readable_topic_labels)

# Human-readable label chosen after reviewing Topic 3
topic_model.set_topic_labels({3: "Russia and Ukraine"})

topic_model.get_topic_info()[["Topic", "Count", "CustomName"]]


,Topic,Count,CustomName
0,-1,238,Outliers
1,0,252,0 · ukraine · russia · russian
2,1,226,1 · star · actor · new
3,2,138,2 · trump · gop · senate
4,3,112,Russia and Ukraine
5,4,87,4 · shooting · school · police
6,5,64,5 · abortion · court · supreme court
7,6,64,6 · covid · covid 19 · 19
8,7,46,7 · beijing · winter · team
9,8,41,8 · dead · missing · crash


## 4. Interpret topics with words articles -- c-TF-IDF scores

The word bars show c-TF-IDF scores: terms that are comparatively distinctive for a topic. The document map then shows how articles are arranged in a separate two-dimensional projection. A plausible topic name should still survive a reading of representative articles.

In [13]:
top_topic_ids = topic_info.loc[topic_info["Topic"] != -1, "Topic"].head(8).tolist()

topic_model.visualize_barchart(
    topics=top_topic_ids,
    n_words=6,
    custom_labels=True,
    width=280,
    height=280,
    title="<b>Distinctive words in the largest topics</b>",
)

In [14]:
topic_model

### Human interpretation step

After reviewing the topic words, Topic 3 is labeled **Russia and Ukraine**. This changes only the display label, not which articles belong to the topic.


In [15]:
# Topic 3 has been revised to a human-readable label above.
# This does not change the clustering; it only changes the displayed name.
topic_model.get_topic_info()[["Topic", "Count", "CustomName"]]


,Topic,Count,CustomName
0,-1,238,Outliers
1,0,252,0 · ukraine · russia · russian
2,1,226,1 · star · actor · new
3,2,138,2 · trump · gop · senate
4,3,112,Russia and Ukraine
5,4,87,4 · shooting · school · police
6,5,64,5 · abortion · court · supreme court
7,6,64,6 · covid · covid 19 · 19
8,7,46,7 · beijing · winter · team
9,8,41,8 · dead · missing · crash


In [16]:
# copy and paste the bar chart code from above ^

## 5. Visualization

### Visualize documents by topic

Each point represents one article. We use headlines as hover text and create a separate two-dimensional UMAP projection for visualization. This projection does not change the fitted BERTopic model or its topic assignments.

In [17]:
news

,document_id,date,year,headline,short_description,authors,link,text
0,news-2022-0001,2022-09-23,2022,23 Of The Funniest Tweets About Cats And Dogs This Week (Sept. 17-23),"""Until you have a dog you don't understand what could be eaten.""",Elyse Wanshel,https://www.huffpost.com/entry/funniest-tweets-cats-dogs-september-17-23_n_632de332e4b0695c1d81dc02,"23 Of The Funniest Tweets About Cats And Dogs This Week (Sept. 17-23). ""Until you have a dog you don't understand wh..."
1,news-2022-0002,2022-09-23,2022,"American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video","He was subdued by passengers and crew when he fled to the back of the aircraft after the confrontation, according to...",Mary Papenfuss,https://www.huffpost.com/entry/american-airlines-passenger-banned-flight-attendant-punch-justice-department_n_632e25...,"American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video. He was subdued by passeng..."
2,news-2022-0003,2022-09-23,2022,Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters,Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new b...,"Carla K. Johnson, AP",https://www.huffpost.com/entry/covid-boosters-uptake-us_n_632d719ee4b087fae6feaac9,Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters. Health experts said it is too early to...
3,news-2022-0004,2022-09-23,2022,The Funniest Tweets From Parents This Week (Sept. 17-23),"""Accidentally put grown-up toothpaste on my toddler’s toothbrush and he screamed like I was cleaning his teeth with ...",Caroline Bologna,https://www.huffpost.com/entry/funniest-parenting-tweets_l_632d7d15e4b0d12b5403e479,"The Funniest Tweets From Parents This Week (Sept. 17-23). ""Accidentally put grown-up toothpaste on my toddler’s toot..."
4,news-2022-0005,2022-09-22,2022,Cleaner Was Dead In Belk Bathroom For 4 Days Before Body Found: Police,The 63-year-old woman was seen working at the South Carolina store on Thursday. She was found dead Monday after her ...,NaN,https://www.huffpost.com/entry/belk-worker-found-dead-columbiana-centre-bathroom_n_632c5f8ce4b0572027b0251d,Cleaner Was Dead In Belk Bathroom For 4 Days Before Body Found: Police. The 63-year-old woman was seen working at th...
...,...,...,...,...,...,...,...,...
1393,news-2022-1394,2022-01-01,2022,Austrian Holocaust Survivor 'Mrs. Gertrude' Dies At 94,Gertrude Pressburger became famous during Austria's 2016 presidential campaign after warning of the dangers of the f...,"Kirsten Grieshaber, AP",https://www.huffpost.com/entry/ap-eu-austria-obit-gertrude-pressburger_n_61d0aaa6e4b0bcd219540b75,Austrian Holocaust Survivor 'Mrs. Gertrude' Dies At 94. Gertrude Pressburger became famous during Austria's 2016 pre...
1394,news-2022-1395,2022-01-01,2022,New York City Rings In 2022 With Times Square Ball Drop Amid Smaller Crowd And COVID,"The city’s incoming mayor, Eric Adams, took his oath of office on the scene shortly afterward.","By BOBBY CAINA CALVAN, AP",https://www.huffpost.com/entry/times-square-ball-drop-eric-adams-oath_n_61cfe456e4b0c7d8b8a4586e,"New York City Rings In 2022 With Times Square Ball Drop Amid Smaller Crowd And COVID. The city’s incoming mayor, Eri..."
1395,news-2022-1396,2022-01-01,2022,Rep. Alexandria Ocasio-Cortez Rips 'Creepy Weirdo' Trump Adviser For Fixating On Boyfriend's Feet,"The congresswoman fired back at Republican Steve Cortes, turning the tables on his bizarre crack about her trip to F...",Mary Papenfuss,https://www.huffpost.com/entry/alexandria-ocasio-cortez-steve-cortes-florida-sandals-feet_n_61cf9f2de4b04b42ab7441ee,Rep. Alexandria Ocasio-Cortez Rips 'Creepy Weirdo' Trump Adviser For Fixating On Boyfriend's Feet. The congresswoman...
1396,news-2022-1397,2022-01-01,2022,Some Of The Biggest Lies Being Spread About The Jan. 6 Insurrection,The events from last Jan. 6 in Washington unfolded on live television but they've since been up for deb

In [18]:
from sklearn.base import clone

titles = news["headline"].tolist()  # Converts headlines to a list for plot labels.

umap_2d = clone(umap_model).set_params(n_components=2) # more about `n_components`: check out https://umap-learn.readthedocs.io/en/latest/parameters.html#n-components if you're interested
# Copies all UMAP settings but changes the output to two dimensions.

reduced_embeddings = umap_2d.fit_transform(embeddings)
# Creates 2D coordinates without changing the original 5D clustering model.

In [19]:
reduced_embeddings

array([[ 7.3633084 ,  2.629793  ],
       [ 6.558786  ,  0.6989282 ],
       [ 4.8647714 ,  3.2299428 ],
       ...,
       [ 9.108941  , -0.92002517],
       [ 8.806761  , -0.93457574],
       [ 4.6883245 ,  3.443581  ]], shape=(1398, 2), dtype=float32)

In [20]:
"""
topic_model.visualize_documents(
    titles,                                  # Uses headlines to identify documents.
    reduced_embeddings=reduced_embeddings,  # Plots the precomputed 2D coordinates.
    custom_labels=True,                      # Uses the custom topic labels.
    hide_annotations=True,                   # Hides labels drawn over the plot.
)
# this code ^ prints out harmless warning messages -- I'm just suppressing the warning below.
"""


import warnings
from pandas.errors import SettingWithCopyWarning

with warnings.catch_warnings():
    warnings.simplefilter("ignore", SettingWithCopyWarning)

    fig = topic_model.visualize_documents(
        titles,
        reduced_embeddings=reduced_embeddings,
        custom_labels=True,
        hide_annotations=True,
    )

fig

> **Interpretation caution:** Nearby points have similar positions in this two-dimensional UMAP projection, but the distances are approximate. Hover over points and return to the article text before drawing conclusions about a topic or boundary.

### topis and hierarchy

This section follows the shared *BERTopic Best Practices* notebook closely. Its two go-to topic visuals are:

- `topic_model.visualize_topics()`
- `topic_model.visualize_hierarchy()`

The same calls are used below with the readable labels created earlier.

In [21]:
topic_model.visualize_topics(custom_labels=True)

### Read the intertopic distance map

- Circle size reflects topic size.
- Nearby circles have more similar topic representations in the two-dimensional view.
- Separated circles suggest more distinct representations.

In [22]:
topic_model.visualize_hierarchy(custom_labels=True)

### Read the topic hierarchy

The dendrogram joins the most similar topic representations first. Branches that merge later are less similar under this representation.

## 6. Export the Results

In [23]:
document_info = topic_model.get_document_info(documents)
# Creates a DataFrame with one row for each document.
# It combines each original document with the topic assignment saved during
# fit_transform(), including the topic number, topic name, custom label,
# top keywords, assignment probability, and whether the document is a
# representative example of its topic.
# The documents must be in the same order as those used to train the model.

results = news.copy()  # Copies the original dataset.
results["topic"] = document_info["Topic"]  # Adds each document’s assigned topic ID.
results["topic_label"] = document_info["CustomName"]  # Adds its readable topic label.
results["topic_keywords"] = document_info["Top_n_words"]  # Adds its topic’s main keywords.
results["membership_strength"] = document_info["Probability"]  # Adds HDBSCAN membership strength.
results["representative_document"] = document_info["Representative_document"]  # Marks representative examples.

results[["headline", "topic", "topic_label"]].head()  # Displays the first five selected results.

results.topic.value_counts()

topic
 0     252
-1     238
 1     226
 2     138
 3     112
 4      87
 5      64
 6      64
 7      46
 8      41
 9      38
 10     34
 11     33
 12     25
Name: count, dtype: int64

In [24]:
results

,document_id,date,year,headline,short_description,authors,link,text,topic,topic_label,topic_keywords,membership_strength,representative_document
0,news-2022-0001,2022-09-23,2022,23 Of The Funniest Tweets About Cats And Dogs This Week (Sept. 17-23),"""Until you have a dog you don't understand what could be eaten.""",Elyse Wanshel,https://www.huffpost.com/entry/funniest-tweets-cats-dogs-september-17-23_n_632de332e4b0695c1d81dc02,"23 Of The Funniest Tweets About Cats And Dogs This Week (Sept. 17-23). ""Until you have a dog you don't understand wh...",10,10 · tweets · week · june,tweets - week - june - 20 - parents - dogs - cat - 23,1.000000,False
1,news-2022-0002,2022-09-23,2022,"American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video","He was subdued by passengers and crew when he fled to the back of the aircraft after the confrontation, according to...",Mary Papenfuss,https://www.huffpost.com/entry/american-airlines-passenger-banned-flight-attendant-punch-justice-department_n_632e25...,"American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video. He was subdued by passeng...",-1,Outliers,police - twitter - man - said - new - water - people - woman,0.000000,False
2,news-2022-0003,2022-09-23,2022,Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters,Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new b...,"Carla K. Johnson, AP",https://www.huffpost.com/entry/covid-boosters-uptake-us_n_632d719ee4b087fae6feaac9,Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters. Health experts said it is too early to...,6,6 · covid · covid 19 · 19,covid - covid 19 - 19 - cases - health - mandate - new - city,1.000000,False
3,news-2022-0004,2022-09-23,2022,The Funniest Tweets From Parents This Week (Sept. 17-23),"""Accidentally put grown-up toothpaste on my toddler’s toothbrush and he screamed like I was cleaning his teeth with ...",Caroline Bologna,https://www.huffpost.com/entry/funniest-parenting-tweets_l_632d7d15e4b0d12b5403e479,"The Funniest Tweets From Parents This Week (Sept. 17-23). ""Accidentally put grown-up toothpaste on my toddler’s toot...",10,10 · tweets · week · june,tweets - week - june - 20 - parents - dogs - cat - 23,1.000000,False
4,news-2022-0005,2022-09-22,2022,Cleaner Was Dead In Belk Bathroom For 4 Days Before Body Found: Police,The 63-year-old woman was seen working at the South Carolina store on Thursday. She was found dead Monday after her ...,NaN,https://www.huffpost.com/entry/belk-worker-found-dead-columbiana-centre-bathroom_n_632c5f8ce4b0572027b0251d,Cleaner Was Dead In Belk Bathroom For 4 Days Before Body Found: Police. The 63-year-old woman was seen working at th...,8,8 · dead · missing · crash,dead - missing - crash - killed - injured - people - dozens - days,1.000000,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1393,news-2022-1394,2022-01-01,2022,Austrian Holocaust Survivor 'Mrs. Gertrude' Dies At 94,Gertrude Pressburger became famous during Austria's 2016 presidential campaign after warning of the dangers of the f...,"Kirsten Grieshaber, AP",https://www.huffpost.com/entry/ap-eu-austria-obit-gertrude-pressburger_n_61d0aaa6e4b0bcd219540b75,Austrian Holocaust Survivor 'Mrs. Gertrude' Dies At 94. Gertrude Pressburger became famous during Austria's 2016 pre...,0,0 · ukraine · russia · russian,ukraine - russia - russian - biden - war - president - said - day,0.484141,False
1394,news-2022-1395,2022-01-01,2022,New York City Rings In 2022 With Times Square Ball Drop Amid Smaller Crowd And COVID,"The city’s incoming mayor, Eric Adams, took his oath of office on the scene shortly afterward.","By BOBBY CAINA CALVAN, AP",https://www.huffpost.com/entry/times-square-ball-drop-eric-adams-oath_n_61cfe456e4b0c7d8b8a4586e,"New York City Rings In 2022 With Times Square Ball Drop Amid Smaller Crowd And COVID. The city’s incoming mayor, Eri...",9,9 ·

### Detailed `c-TF-IDF score` values for a selected topic

The **c-TF-IDF score** measures how important a word or phrase is to a
particular topic.

A high score means the term:

- appears often in that topic, and
- helps distinguish that topic from the other topics.

It is an importance score—not a probability or a raw word count.

In [25]:
TOPIC_TO_INSPECT = top_topic_ids[0]# Selects the first topic ID from the list of top topics.

topic_words = pd.DataFrame(
    topic_model.get_topic(TOPIC_TO_INSPECT),
    # Gets the topic's important words or phrases and their c-TF-IDF scores.

    columns=["word or phrase", "c-TF-IDF score"],
    # Gives the two resulting columns readable names.
)

representative_articles = results[
    (results["topic"] == TOPIC_TO_INSPECT)
    # Keeps only articles assigned to the selected topic.

    & (results["representative_document"])
    # Further limits the results to articles BERTopic selected as good
    # examples of that topic.
]

display(topic_words.head(8))
display(representative_articles[["headline", "short_description"]])

,word or phrase,c-TF-IDF score
0,ukraine,0.099414
1,russia,0.072270
2,russian,0.069006
3,biden,0.050772
4,war,0.048997
5,president,0.041254
6,said,0.025564
7,day,0.023977


,headline,short_description
1009,Ukraine President Volodymyr Zelenskyy To Appeal To U.S. Lawmakers For More Help Against Russia,Ukraine’s president seeks help to “close the skies over Ukraine to Russian missiles and planes.”
1051,Biden Reportedly Set To Ban Russian Oil Imports Over Ukraine War,The move follows pleas by Ukrainian President Volodmyr Zelenskyy to U.S. and Western officials to cut off the imports.
1069,Kamala Harris To Travel To Poland And Romania Amid Russian Invasion Of Ukraine,The vice president is meeting with NATO allies to coordinate their support for Ukraine in the face of Russia’s “unpr...


### Your turn: obtain a c-TF-IDF score for a different topic

## 7. Inspect the outlier group

Outliers are part of density-based clustering. They preserve evidence that some articles do not fit the dense groups well under the current settings.

In [26]:
outliers = results[results["topic"] == -1]

outliers.sample(6, random_state=42)[
    ["headline", "short_description", "topic_keywords"]
]

,headline,short_description,topic_keywords
697,Man Pops The Question After Engagement Ring Survives Car Fire,Myers Hart said he took the incident as a “sign” to propose.,police - twitter - man - said - new - water - people - woman
102,Citing Imminent Danger Cloudflare Drops Hate Site Kiwi Farms,Cloudflare CEO Matthew Prince had previously resisted calls to block the site.,police - twitter - man - said - new - water - people - woman
1284,Whoopi Goldberg Slams Bill Maher’s ‘Flippant’ COVID-19 Commentary,"The ""View"" co-host said the ""Real Time"" comedian's gags were ""not really funny to people who have lost their kids"" t...",police - twitter - man - said - new - water - people - woman
763,Judge Tosses Part Of Blac Chyna Case Against Kim Kardashian,Jurors went home Friday without reaching a verdict after their first full day of deliberations.,police - twitter - man - said - new - water - people - woman
43,Racism Seen As Root Of Water Crisis In Mississippi Capital,"A boil-water advisory has been lifted for Mississippi's capital, and the state will stop handing out free bottled wa...",police - twitter - man - said - new - water - people - woman
999,"Biden COVID Coordinators Leaving, To Be Replaced By Ashish Jha","Jeff Zients and his deputy Natalie Quillian will leave the administration next month. Jha, the dean of the Brown Uni...",police - twitter - man - said - new - water - people - woman


### Your turn: interpret one outlier

Select one row.

- Is it genuinely unusual for this collection?
- Could it plausibly belong to a discovered topic?
- What evidence would justify leaving it unassigned?

Do not force assignment merely to make the output look complete.